In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🔌 Serverless Patterns & Optimization Guide

**Building scalable, cost-effective serverless applications with optimization techniques and best practices**

This guide covers:
- Cold start optimization
- Function composition patterns
- State management in serverless
- Concurrency and throttling
- Cost optimization
- Monitoring and debugging
- Error handling and retry logic
- Serverless architecture patterns
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Cold Start Mitigation

### Warm Start Strategies
```python
from dataclasses import dataclass
from typing import Dict, Optional, Callable, List
from datetime import datetime, timedelta
import time

@dataclass
class FunctionMetrics:
    """Metrics for serverless function"""
    function_name: str
    cold_start_duration_ms: Optional[int] = None
    warm_start_duration_ms: Optional[int] = None
    is_cold_start: bool = False
    execution_time_ms: int = 0
    memory_used_mb: int = 0
    billed_duration_ms: int = 0

class ColdStartOptimizer:
    """Optimize function cold starts"""
    
    def __init__(self):
        self.functions: Dict[str, Dict] = {}
        self.metrics_history: List[FunctionMetrics] = []
    
    def register_function(self, function_name: str, config: Dict):
        """Register function for optimization"""
        
        self.functions[function_name] = {
            'name': function_name,
            'language': config.get('language'),
            'memory_mb': config.get('memory_mb', 128),
            'ephemeral_storage_mb': config.get('ephemeral_storage_mb', 512),
            'layers': config.get('layers', []),
            'initialized': False,
            'last_invoked': None
        }
    
    def enable_provisioned_concurrency(self, function_name: str, concurrent_instances: int) -> bool:
        """Enable provisioned concurrency to keep instances warm"""
        
        if function_name not in self.functions:
            return False
        
        self.functions[function_name]['provisioned_concurrency'] = concurrent_instances
        
        return True
    
    def configure_ephemeral_storage(self, function_name: str, storage_mb: int) -> bool:
        """Increase ephemeral storage for faster downloads"""
        
        if function_name not in self.functions:
            return False
        
        self.functions[function_name]['ephemeral_storage_mb'] = min(storage_mb, 10240)
        
        return True
    
    def optimize_dependencies(self, function_name: str) -> Dict:
        """Recommend dependency optimizations"""
        
        recommendations = {
            'use_lambda_layers': True,
            'reduce_package_size': True,
            'use_compiled_dependencies': True,
            'lazy_load_dependencies': True,
            'use_native_libraries': True
        }
        
        return recommendations
    
    def record_invocation(self, function_name: str, metrics: FunctionMetrics):
        """Record function invocation metrics"""
        
        self.metrics_history.append(metrics)
    
    def estimate_monthly_cost(self, function_name: str, monthly_invocations: int) -> Dict:
        """Estimate serverless cost"""
        
        if function_name not in self.functions:
            return {}
        
        func = self.functions[function_name]
        
        # AWS Lambda pricing (simplified)
        request_price = 0.0000002  # per request
        gb_second_price = 0.0000166667  # per GB-second
        
        memory_gb = func['memory_mb'] / 1024
        avg_duration_seconds = 1.0  # Assume 1 second
        
        monthly_requests_cost = monthly_invocations * request_price
        monthly_compute_cost = monthly_invocations * memory_gb * avg_duration_seconds * gb_second_price
        
        # Provisioned concurrency cost (simplified)
        provisioned_cost = 0
        if 'provisioned_concurrency' in func:
            concurrency = func['provisioned_concurrency']
            provisioned_cost = concurrency * memory_gb * 0.015  # $0.015 per GB-hour
        
        total = monthly_requests_cost + monthly_compute_cost + provisioned_cost
        
        return {
            'monthly_requests_cost': round(monthly_requests_cost, 4),
            'monthly_compute_cost': round(monthly_compute_cost, 4),
            'provisioned_concurrency_cost': round(provisioned_cost, 2),
            'total_monthly_cost': round(total, 2)
        }

class WarmStartScheduler:
    """Keep functions warm with scheduled invocations"""
    
    def __init__(self):
        self.schedules: Dict[str, Dict] = {}
    
    def schedule_warmup(self, function_name: str, interval_minutes: int = 5):
        """Schedule periodic warmup invocation"""
        
        self.schedules[function_name] = {
            'interval_minutes': interval_minutes,
            'last_warmup': datetime.now(),
            'enabled': True
        }
    
    def should_invoke_warmup(self, function_name: str) -> bool:
        """Check if warmup invocation needed"""
        
        schedule = self.schedules.get(function_name)
        
        if not schedule or not schedule['enabled']:
            return False
        
        elapsed = (datetime.now() - schedule['last_warmup']).total_seconds() / 60
        
        return elapsed >= schedule['interval_minutes']
    
    def execute_warmup(self, function_name: str) -> bool:
        """Execute warmup invocation"""
        
        if function_name not in self.schedules:
            return False
        
        self.schedules[function_name]['last_warmup'] = datetime.now()
        
        return True
```

### Memory and Dependency Optimization
```python
class DependencyAnalyzer:
    """Analyze and optimize dependencies"""
    
    def __init__(self):
        self.dependencies: Dict[str, Dict] = {}
    
    def add_dependency(self, name: str, size_mb: float, is_required: bool = True):
        """Record dependency"""
        
        self.dependencies[name] = {
            'name': name,
            'size_mb': size_mb,
            'is_required': is_required,
            'lazy_loadable': not is_required
        }
    
    def calculate_package_size(self) -> Dict:
        """Calculate total package size"""
        
        total_size = sum(d['size_mb'] for d in self.dependencies.values())
        required_size = sum(d['size_mb'] for d in self.dependencies.values() if d['is_required'])
        optional_size = total_size - required_size
        
        return {
            'total_mb': round(total_size, 2),
            'required_mb': round(required_size, 2),
            'optional_mb': round(optional_size, 2),
            'can_lazy_load_mb': round(optional_size, 2)
        }
    
    def recommend_optimizations(self) -> List[str]:
        """Recommend optimization strategies"""
        
        recommendations = []
        
        total_size = self.calculate_package_size()
        
        if total_size['total_mb'] > 50:
            recommendations.append("Use Lambda layers to separate dependencies")
        
        if total_size['optional_mb'] > 10:
            recommendations.append("Lazy load optional dependencies")
        
        if any(d['size_mb'] > 10 for d in self.dependencies.values()):
            recommendations.append("Use lightweight alternatives for large dependencies")
        
        return recommendations

class MemoryOptimizer:
    """Optimize memory allocation"""
    
    def __init__(self):
        self.performance_data: List[Dict] = []
    
    def record_performance(self, memory_mb: int, duration_ms: int, throttled: bool = False):
        """Record function performance"""
        
        self.performance_data.append({
            'memory_mb': memory_mb,
            'duration_ms': duration_ms,
            'throttled': throttled,
            'timestamp': datetime.now()
        })
    
    def find_optimal_memory(self) -> int:
        """Find optimal memory allocation"""
        
        if not self.performance_data:
            return 128
        
        # Find memory level with best performance per dollar
        memory_performance = {}
        
        for data in self.performance_data:
            mem = data['memory_mb']
            
            if mem not in memory_performance:
                memory_performance[mem] = {
                    'total_duration': 0,
                    'count': 0,
                    'throttled_count': 0
                }
            
            memory_performance[mem]['total_duration'] += data['duration_ms']
            memory_performance[mem]['count'] += 1
            
            if data['throttled']:
                memory_performance[mem]['throttled_count'] += 1
        
        # Prefer memory level with no throttling
        for mem in sorted(memory_performance.keys()):
            if memory_performance[mem]['throttled_count'] == 0:
                return mem
        
        return max(memory_performance.keys())
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Function Composition & State Management

### Function Composition Patterns
```python
class FunctionComposer:
    """Compose serverless functions"""
    
    def __init__(self):
        self.functions: Dict[str, Callable] = {}
        self.workflows: Dict[str, List[str]] = {}
    
    def register_function(self, name: str, func: Callable):
        """Register function"""
        self.functions[name] = func
    
    def create_pipeline(self, pipeline_name: str, function_sequence: List[str]) -> bool:
        """Create function pipeline"""
        
        # Verify all functions exist
        for func_name in function_sequence:
            if func_name not in self.functions:
                return False
        
        self.workflows[pipeline_name] = function_sequence
        return True
    
    def execute_pipeline(self, pipeline_name: str, initial_input: Dict) -> Optional[Dict]:
        """Execute function pipeline"""
        
        if pipeline_name not in self.workflows:
            return None
        
        result = initial_input
        
        for func_name in self.workflows[pipeline_name]:
            func = self.functions[func_name]
            result = func(result)
            
            if result is None:
                return None
        
        return result

class StateStore:
    """Manage state between function invocations"""
    
    def __init__(self):
        self.state: Dict[str, Dict] = {}
        self.ttl_seconds: Dict[str, int] = {}
    
    def save_state(self, execution_id: str, state: Dict, ttl_seconds: int = 3600):
        """Save execution state"""
        
        self.state[execution_id] = {
            'data': state,
            'created_at': datetime.now(),
            'ttl_seconds': ttl_seconds
        }
        
        self.ttl_seconds[execution_id] = ttl_seconds
    
    def load_state(self, execution_id: str) -> Optional[Dict]:
        """Load execution state"""
        
        if execution_id not in self.state:
            return None
        
        state_entry = self.state[execution_id]
        
        # Check if expired
        age_seconds = (datetime.now() - state_entry['created_at']).total_seconds()
        
        if age_seconds > state_entry['ttl_seconds']:
            del self.state[execution_id]
            return None
        
        return state_entry['data']
    
    def cleanup_expired(self):
        """Remove expired state"""
        
        expired = []
        
        for exec_id, state_entry in self.state.items():
            age_seconds = (datetime.now() - state_entry['created_at']).total_seconds()
            
            if age_seconds > state_entry['ttl_seconds']:
                expired.append(exec_id)
        
        for exec_id in expired:
            del self.state[exec_id]
        
        return len(expired)
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Concurrency & Error Handling

### Concurrency Management
```python
class ConcurrencyController:
    """Control function concurrency"""
    
    def __init__(self, max_concurrent: int = 1000):
        self.max_concurrent = max_concurrent
        self.current_executions = 0
        self.queued_invocations: List[Dict] = []
        self.throttle_count = 0
    
    def can_accept_invocation(self) -> bool:
        """Check if invocation can be accepted"""
        return self.current_executions < self.max_concurrent
    
    def acquire_slot(self) -> bool:
        """Acquire execution slot"""
        
        if self.can_accept_invocation():
            self.current_executions += 1
            return True
        
        self.throttle_count += 1
        return False
    
    def release_slot(self):
        """Release execution slot"""
        if self.current_executions > 0:
            self.current_executions -= 1
    
    def queue_invocation(self, invocation: Dict):
        """Queue invocation if throttled"""
        self.queued_invocations.append(invocation)
    
    def process_queue(self) -> int:
        """Process queued invocations"""
        processed = 0
        
        while self.queued_invocations and self.can_accept_invocation():
            invocation = self.queued_invocations.pop(0)
            self.acquire_slot()
            processed += 1
        
        return processed

class ErrorHandler:
    """Handle function errors with retry logic"""
    
    def __init__(self):
        self.retry_policies: Dict[str, Dict] = {}
        self.error_log: List[Dict] = []
    
    def set_retry_policy(self, function_name: str, max_retries: int = 3, 
                        backoff_base: float = 2.0):
        """Set retry policy for function"""
        
        self.retry_policies[function_name] = {
            'max_retries': max_retries,
            'backoff_base': backoff_base
        }
    
    def calculate_backoff(self, attempt: int, backoff_base: float = 2.0) -> float:
        """Calculate exponential backoff"""
        return min((backoff_base ** attempt) + (0.1 * attempt), 900)  # Max 15 min
    
    def execute_with_retry(self, function_name: str, func: Callable, 
                          *args, **kwargs) -> Optional[Dict]:
        """Execute function with retry logic"""
        
        policy = self.retry_policies.get(function_name, {
            'max_retries': 3,
            'backoff_base': 2.0
        })
        
        max_retries = policy['max_retries']
        backoff_base = policy['backoff_base']
        
        for attempt in range(max_retries + 1):
            try:
                result = func(*args, **kwargs)
                return result
            except Exception as e:
                self.error_log.append({
                    'function': function_name,
                    'attempt': attempt + 1,
                    'error': str(e),
                    'timestamp': datetime.now()
                })
                
                if attempt < max_retries:
                    backoff_seconds = self.calculate_backoff(attempt, backoff_base)
                    time.sleep(backoff_seconds)
                else:
                    raise
        
        return None
    
    def get_error_summary(self) -> Dict:
        """Get error summary statistics"""
        
        if not self.error_log:
            return {'total_errors': 0}
        
        function_errors = {}
        
        for error in self.error_log:
            func = error['function']
            
            if func not in function_errors:
                function_errors[func] = 0
            
            function_errors[func] += 1
        
        return {
            'total_errors': len(self.error_log),
            'by_function': function_errors
        }
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Serverless Patterns & Optimization Checklist

✅ **Cold Start Optimization**
- [ ] Provisioned concurrency enabled
- [ ] Lambda layers configured
- [ ] Dependencies optimized
- [ ] Memory allocation tuned
- [ ] Warmup strategy implemented

✅ **Function Design**
- [ ] Single responsibility principle
- [ ] Composition patterns used
- [ ] State management strategy
- [ ] Error handling robust
- [ ] Timeouts configured

✅ **Concurrency & Performance**
- [ ] Concurrency limits set
- [ ] Throttling monitored
- [ ] Retry logic implemented
- [ ] Circuit breakers added
- [ ] Performance baseline

✅ **Cost Management**
- [ ] Cost tracking enabled
- [ ] Memory optimized
- [ ] Unused functions eliminated
- [ ] Reserved concurrency considered
- [ ] Cost alerts configured

✅ **Operational Excellence**
- [ ] Logging configured
- [ ] Distributed tracing
- [ ] Alarms and alerts
- [ ] Runbooks documented
- [ ] Regular reviews scheduled
</VSCode.Cell>
```